# Lab 1. Quantum Teleportation That Preserves Entanglement

## Introduction

In this lab, you will build a circuit that <strong>transfers quantum information while preserving entanglement</strong>. You will then write checks to determine whether the transfer worked.

Suppose that two qubits, R and Q, have been prepared in the following Bell state:

$$
|\Phi^+\rangle_{RQ}
=\frac{|0\rangle_R|0\rangle_Q+|1\rangle_R|1\rangle_Q}{\sqrt2}.
$$

The subscripts R,Q identify the <strong>joint state of the two qubits</strong>. For example, $|0\rangle_R|0\rangle_Q$ is the component in which both R and Q are 0. Alice holds Q. Our aim is to transfer the quantum information in Q to Bob's qubit B while leaving R untouched.

If the transfer succeeds, <strong>R and B must share the same Bell state that R and Q originally shared.</strong> The target state after the transfer is therefore

$$
|\Phi^+\rangle_{RB}
=\frac{|0\rangle_R|0\rangle_B+|1\rangle_R|1\rangle_B}{\sqrt2}
$$

The change between the two expressions is that B takes the place of Q. Although we transfer the quantum information in Q, judging success also requires checking that its entanglement with R has been preserved.

In this exercise, Q cannot be sent directly to Bob. Alice and Bob instead share a separate Bell pair A,B in advance: Alice holds A, and Bob holds B. <strong>R,Q form the input state we want to preserve, whereas A,B provide the resource for teleportation.</strong> The experiment therefore uses four physical qubits in total.

Alice applies operations to her qubits Q and A, then measures both. We denote the Q outcome by $m_Q$ and the A outcome by $m_A$, recording each as one classical bit. The joint state of R and B depends on these outcomes, so Bob receives the two bits and applies the appropriate operations to B. These operations are called <strong>corrections</strong>.

The transfer and verification proceed as follows.

| Stage | Qubits or information used | Task |
|---|---|---|
| Prepare the input and resource | R,Q and A,B | Prepare the input Bell state and the separate Bell state used for teleportation. |
| Alice's operations and measurements | Q,A | Apply the two-qubit operations and record outcomes $m_Q,m_A$. |
| Classical communication and Bob's correction | Received $m_Q,m_A$ and B | Apply operations to B according to the received record. |
| Verify the transfer | R,B | Compare their corrected joint state with the original state of R,Q. |

No teleportation operation acts on R. We will check whether the same correction rules restore the joint state of R and B even when the input state changes.

After completing the circuit, you will verify its output. <strong>Checking each qubit's measurement results separately is insufficient.</strong> Even when each qubit has the same proportions of 0 and 1 outcomes, the correlations between their results can differ. You will therefore analyze the paired outcomes of R and B and learn how to check whether their joint state is preserved by the transfer.

Later, you will examine a report that analyzes only selected measurement outcomes, and a situation in which Alice's measurement record reaches Bob incorrectly. Reuse the checks you have built to explain how the transfer changes in each case.

## How to Use This Notebook

Each section follows the sequence <strong>purpose and notation → key equations and examples → exercises</strong>. Understand what the equations mean, predict the result, and then compare your prediction with the code.

Run the code cells from top to bottom. Execute the provided tools as well, but you do not need to read their entire implementation. Each problem explains how to use the tools it needs.

1. <strong>Write your prediction and reasoning.</strong> First identify the input, the target, and the information available to you.
2. <strong>Complete the TODO.</strong> Replace `raise NotImplementedError` between `TODO BEGIN` and `TODO END` with your solution, keeping the function name and parameters unchanged.
3. <strong>Run the answer cell and its check cell.</strong> After editing code, rerun both cells and any later experiments that use the function.
4. <strong>Compare the results with your predictions and explain them.</strong> If they differ from your prediction, review your assumptions and calculations, and explain your reasoning in the written answer field.

<strong>Write your predictions before running the experiments.</strong> In the student notebook, `RUN_OBSERVATIONS` initially equals `False`. After writing your prediction, change it to `True` in the configuration cell after the provided tools, run that cell, and then run the relevant experiment cell. Once enabled, the setting also applies to later experiments, so write each prediction and its reasoning before running that experiment. You can change it back to `False` when needed.

The provided function checks run independently of this setting. Completing the TODOs does not automatically enable the experiment cells. The final application is a written exercise, so it has no separate execution setting or function TODO.

Check cells display the following statuses.

- <strong>Incomplete:</strong> A TODO has not yet been implemented.
- <strong>Failed:</strong> An error occurred or the result does not satisfy the problem's requirements. Read the accompanying message and revise your code.
- <strong>Passed:</strong> The provided code checks passed.

Written explanations are not graded by the automatic checks. After checking your code, make sure you have also completed every explanation.

<strong>You may use AI to study in this lab.</strong> Ask for conceptual explanations, check calculations, or investigate error messages. Run any AI-suggested code yourself and verify that it produces the expected result. In written answers, use the results you obtain to explain in your own words why the operations or verification methods are appropriate.

## Set Up the Environment

This notebook is checked with Python 3.12, Qiskit 2.3.1, Qiskit Aer 0.17.2, and NumPy 2.2.6. For local use, install the accompanying `requirements.txt` and select that environment as your Jupyter kernel.

If required packages are missing in Colab or another environment, run the following command in a new code cell, restart the runtime, and then run the notebook from the top.

```python
%pip install qiskit==2.3.1 qiskit-aer==0.17.2 numpy==2.2.6
```

Run the provided tool cells below as well. Long code may appear collapsed. Follow the explanations and equations in each problem, and inspect the results through tables and text circuit diagrams.

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python", sys.version.split()[0])
for package in ("qiskit", "qiskit-aer", "numpy"):
    try:
        print(package, version(package))
    except PackageNotFoundError as error:
        raise RuntimeError("Install the required packages using the environment setup instructions above.") from error

### Provided Tools

Run the next two cells to prepare the branch calculations and measurement experiments. Write your own solutions in the TODO cells for each problem.

In [ ]:
"""Provided tools: circuit branches, joint-state checks, and measurement samples.

Descriptions and returned arrays use R,B order. Qiskit's internal
little-endian arrays are converted at this module's input/output boundaries.
This file does not contain the student's correction rule.
"""

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Barrier, Clbit, IfElseOp
from qiskit.circuit.classical import expr
from qiskit.circuit.library import IGate, XGate, ZGate
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator


SHOTS = 4096
SEED = 2026
ATOL = 1e-9
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
Z_ON_B = np.kron(I2, Z)
PHI_PLUS = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
PHI_MINUS = np.array([1, 0, 0, -1], dtype=complex) / np.sqrt(2)
PHASE_INPUT = np.array([1, 0, 0, 1j], dtype=complex) / np.sqrt(2)
MIX_RHO = np.diag([0.5, 0, 0, 0.5]).astype(complex)
RB_LABELS = ("00", "01", "10", "11")
BASES = ("XX", "YY", "ZZ")
_H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
_SDG = np.diag([1, -1j])
_PAULIS = {"X": X, "Y": Y, "Z": Z}
_BASIS_ROTATIONS = {"X": _H, "Y": _H @ _SDG, "Z": I2}


def _state_vector(psi, size=4):
    value = np.asarray(psi, dtype=complex)
    if value.shape != (size,) or not np.all(np.isfinite(value)):
        raise ValueError(f"A finite state vector of length {size} is required.")
    if not np.isclose(np.vdot(value, value).real, 1.0, atol=ATOL, rtol=0):
        raise ValueError("The state vector must be normalized.")
    return value.copy()


def _density_matrix(rho):
    value = np.asarray(rho, dtype=complex)
    if value.shape != (4, 4) or not np.all(np.isfinite(value)):
        raise ValueError("A finite 4×4 density matrix in R,B order is required.")
    if not np.allclose(value, value.conj().T, atol=ATOL, rtol=0):
        raise ValueError("The density matrix must equal its conjugate transpose.")
    if not np.isclose(np.trace(value), 1.0, atol=ATOL, rtol=0):
        raise ValueError("A normalized density matrix must have trace 1.")
    if np.linalg.eigvalsh(value).min() < -ATOL:
        raise ValueError("Density-matrix eigenvalues cannot be negative.")
    return value.copy()


def density(psi):
    """Convert a normalized two-qubit vector into a density matrix in R,B order."""
    value = _state_vector(psi)
    return np.outer(value, value.conj())


def fidelity(rho, target=PHI_PLUS):
    """Calculate F = <target|rho|target> for a pure target state."""
    value = _density_matrix(rho)
    psi = _state_vector(target)
    return float(np.vdot(psi, value @ psi).real)


def correlations(rho):
    """Return the three same-axis joint expectation values."""
    value = _density_matrix(rho)
    return {axis + axis: float(np.trace(value @ np.kron(P, P)).real)
            for axis, P in _PAULIS.items()}


def local_expectations(rho):
    """X·Y·Z expectations for measurements of R alone and B alone."""
    value = _density_matrix(rho)
    return {
        "R": {a: float(np.trace(value @ np.kron(P, I2)).real)
              for a, P in _PAULIS.items()},
        "B": {a: float(np.trace(value @ np.kron(I2, P)).real)
              for a, P in _PAULIS.items()},
    }


def z_distribution(rho):
    """Return exact joint Z probabilities keyed by R,B outcome strings."""
    value = _density_matrix(rho)
    probabilities = np.real(np.diag(value))
    return {label: float(p) for label, p in zip(RB_LABELS, probabilities)}


def _teleport_prefix(psi):
    psi = _state_vector(psi)
    qc = QuantumCircuit(4)
    # Match the explanation's |RQ> order to Qiskit's |q1 q0> order.
    qiskit_payload = psi.reshape(2, 2).T.reshape(4)
    qc.initialize(qiskit_payload, [0, 1])
    qc.h(2)
    qc.cx(2, 3)
    qc.cx(1, 2)
    qc.h(1)
    return qc


def uncorrected_branches(psi=PHI_PLUS):
    """Evaluate the actual four-qubit circuit and project onto Alice's four outcomes."""
    state = Statevector.from_instruction(_teleport_prefix(psi)).data
    rows = []
    for mQ in (0, 1):
        for mA in (0, 1):
            raw = np.array([state[r + 2*mQ + 4*mA + 8*b]
                            for r in (0, 1) for b in (0, 1)], dtype=complex)
            probability = float(np.vdot(raw, raw).real)
            if probability <= 0:
                raise ValueError("With this lab's Bell resource, all four measurement branches have positive probability.")
            vector = raw / np.sqrt(probability)
            rows.append({"mQ": mQ, "mA": mA, "probability": probability,
                         "raw_state": raw, "state": vector,
                         "rho": np.outer(vector, vector.conj())})
    return rows


def _decoder_matrix(block, qmap, cmap, bits):
    """Follow if_test branches to obtain the one-qubit operation for a classical input."""
    try:
        phase = float(block.global_phase)
    except (TypeError, ValueError) as exc:
        raise ValueError("The correction circuit cannot contain unbound parameters.") from exc
    if not np.isfinite(phase):
        raise ValueError("The global phase must be finite.")
    result = np.exp(1j * phase) * I2
    for instruction in block.data:
        op = instruction.operation
        qindices = [qmap[block.find_bit(q).index] for q in instruction.qubits]
        cindices = [cmap[block.find_bit(c).index] for c in instruction.clbits]
        if isinstance(op, IfElseOp):
            condition = op.condition
            if (not isinstance(condition, tuple) or len(condition) != 2
                    or not isinstance(condition[0], Clbit)
                    or not isinstance(condition[1], (bool, int, np.integer))
                    or int(condition[1]) not in (0, 1)):
                raise ValueError("Use an if_test condition comparing one classical bit with 0 or 1.")
            cond_index = cmap[block.find_bit(condition[0]).index]
            branches = []
            # Also validate operations in branches that are not selected.
            for child in op.blocks:
                if child is None:
                    branches.append(I2)
                else:
                    if child.num_qubits != len(qindices) or child.num_clbits != len(cindices):
                        raise ValueError("Check the bit connections in the conditional circuit.")
                    branches.append(_decoder_matrix(child, qindices, cindices, bits))
            chosen = 0 if bits[cond_index] == int(condition[1]) else 1
            current = branches[chosen] if chosen < len(branches) else I2
        elif isinstance(op, Barrier):
            if any(q != 0 for q in qindices) or cindices:
                raise ValueError("Only Bob's local qubit may be used.")
            current = I2
        elif isinstance(op, (XGate, ZGate, IGate)):
            if qindices != [0] or cindices:
                raise ValueError("Apply gates only to Bob's q0.")
            current = X if isinstance(op, XGate) else Z if isinstance(op, ZGate) else I2
        else:
            raise ValueError(f"Use only X, Z, and if_test in the correction circuit. Found operation: {op.name}")
        result = current @ result
    return result


def decoder_unitaries(decoder):
    """Evaluate the student's circuit on its four classical inputs. This is not a security sandbox."""
    if not isinstance(decoder, QuantumCircuit):
        raise TypeError("bob_decoder() must return a QuantumCircuit.")
    if decoder.num_qubits != 1 or decoder.num_clbits != 2:
        raise ValueError("Bob's returned circuit must be QuantumCircuit(1, 2).")
    return {(a, b): _decoder_matrix(decoder, [0], [0, 1], (a, b))
            for a in (0, 1) for b in (0, 1)}


def corrected_branches(decoder, psi=PHI_PLUS, flip=0):
    """Evaluate correction with the received Q record in c0 and the accurate A record in c1."""
    if flip not in (0, 1):
        raise ValueError("flip must be 0 or 1.")
    unitaries = decoder_unitaries(decoder)
    rows = []
    for row in uncorrected_branches(psi):
        received = row["mQ"] ^ int(flip)
        U = np.kron(I2, unitaries[received, row["mA"]])
        vector = U @ row["state"]
        rows.append({**row, "receivedQ": received, "e": int(flip),
                     "state": vector, "rho": np.outer(vector, vector.conj())})
    return rows


def _error_probability(p):
    value = float(p)
    if not np.isfinite(value) or not 0 <= value <= 1:
        raise ValueError("Error probability p must be between 0 and 1 inclusive.")
    return value


def noisy_branches(decoder, p, psi=PHI_PLUS):
    """Alice's four branches × two transmission-error cases. Keep zero-probability rows too."""
    p = _error_probability(p)
    rows = []
    for flip, weight in ((0, 1-p), (1, p)):
        for row in corrected_branches(decoder, psi, flip):
            rows.append({**row, "alice_probability": row["probability"],
                         "probability": row["probability"] * weight})
    return rows


def weighted_density(rows):
    """Probability-average and normalize the supplied rows. Subsets are also supported."""
    rows = list(rows)
    if not rows:
        raise ValueError("There are no branches to average.")
    probabilities = np.array([row["probability"] for row in rows], dtype=float)
    if np.any(~np.isfinite(probabilities)) or np.any(probabilities < 0):
        raise ValueError("Branch probabilities must be finite and nonnegative.")
    total = float(probabilities.sum())
    if total <= 0:
        raise ValueError("The selected branches have total probability zero.")
    result = sum(p * _density_matrix(row["rho"])
                 for p, row in zip(probabilities, rows)) / total
    return np.asarray(result, dtype=complex)


def decoder_from_table(table):
    """Convert each received record's ordered X/Z list into an actual if_test circuit."""
    keys = {(a, b) for a in (0, 1) for b in (0, 1)}
    if not isinstance(table, dict) or set(table) != keys:
        raise ValueError("The table requires four (received Q, A) keys: (0,0), (0,1), (1,0), (1,1).")
    qc = QuantumCircuit(1, 2)
    for received, mA in sorted(keys):
        operations = table[received, mA]
        if not isinstance(operations, (tuple, list)) or any(g not in ("x", "z") for g in operations):
            raise ValueError("Each value is a tuple/list of 'x' and 'z' in execution order. Use () for no operation.")
        if operations:
            with qc.if_test((qc.clbits[0], received)):
                with qc.if_test((qc.clbits[1], mA)):
                    for gate in operations:
                        getattr(qc, gate)(0)
    return qc


def missing_z_decoder():
    """Provided variant for Problem 3: keep X correction and omit Z correction."""
    qc = QuantumCircuit(1, 2)
    with qc.if_test((qc.clbits[1], 1)):
        qc.x(0)
    return qc


def _measurement_probabilities(rho, basis):
    if basis not in BASES:
        raise ValueError("The measurement setting must be XX, YY, or ZZ.")
    value = _density_matrix(rho)
    U = np.kron(_BASIS_ROTATIONS[basis[0]], _BASIS_ROTATIONS[basis[1]])
    probabilities = np.diag(U @ value @ U.conj().T).real
    # Remove only floating-point rounding artifacts from a physical density matrix.
    probabilities = np.maximum(probabilities, 0)
    return probabilities / probabilities.sum()


def _shot_count(shots):
    if isinstance(shots, (bool, np.bool_)) or not isinstance(shots, (int, np.integer)) or shots <= 0:
        raise ValueError("shots must be a positive integer.")
    return int(shots)


def sample_state(rho, basis, shots=SHOTS, seed=SEED):
    """Sample exact Born probabilities with NumPy. Returned keys use R,B order."""
    shots = _shot_count(shots)
    counts = np.random.default_rng(seed).multinomial(shots, _measurement_probabilities(rho, basis))
    return {label: int(n) for label, n in zip(RB_LABELS, counts) if n}


def correlator(counts):
    """Sample mean of +1 for equal R,B outcomes and -1 for different outcomes."""
    if not isinstance(counts, dict) or any(k not in RB_LABELS for k in counts):
        raise ValueError("counts keys must be two-bit strings in R,B order.")
    if any(isinstance(n, (bool, np.bool_)) or not isinstance(n, (int, np.integer)) or n < 0
           for n in counts.values()):
        raise ValueError("Observed counts must be nonnegative integers.")
    total = sum(counts.values())
    if total == 0:
        raise ValueError("Cannot estimate a correlation without selected trials.")
    return float(sum((1 if key[0] == key[1] else -1) * n
                     for key, n in counts.items()) / total)


def _selection(keep):
    if keep is None:
        return lambda row: row["mQ"] == 0
    if keep == "all":
        return lambda row: True
    if callable(keep):
        return keep
    raise ValueError("Use 'all' or a predicate accepting a branch row for keep.")


def _finish_sample(records, shots, selector, method):
    all_counts = {key: 0 for key in RB_LABELS}
    kept_counts = {key: 0 for key in RB_LABELS}
    for row in records:
        key = str(row["r"]) + str(row["b"])
        all_counts[key] += row["count"]
        if selector(row):
            kept_counts[key] += row["count"]
    kept = sum(kept_counts.values())
    return {"counts_all": {k: v for k, v in all_counts.items() if v},
            "counts_keep": {k: v for k, v in kept_counts.items() if v},
            "kept": kept, "total": shots, "records": records,
            "selection_rate": kept / shots, "method": method}


def sample_ensemble(rows, shots=SHOTS, seed=SEED, keep=None):
    """Draw fresh samples per setting, recording the classical branch and verification outcomes for each trial."""
    shots = _shot_count(shots)
    rows = list(rows)
    probabilities = np.array([row["probability"] for row in rows], dtype=float)
    if (not rows or np.any(~np.isfinite(probabilities)) or np.any(probabilities < 0)
            or not np.isclose(probabilities.sum(), 1, atol=ATOL, rtol=0)):
        raise ValueError("Supply branch rows whose total probability is 1.")
    probabilities = probabilities / probabilities.sum()
    selector = _selection(keep)
    result = {}
    for index, basis in enumerate(BASES):
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), index]))
        branch_counts = rng.multinomial(shots, probabilities)
        records = []
        for row, n in zip(rows, branch_counts):
            if n == 0:
                continue
            outcomes = rng.multinomial(n, _measurement_probabilities(row["rho"], basis))
            for label, count in zip(RB_LABELS, outcomes):
                if count:
                    records.append({"mQ": int(row["mQ"]), "mA": int(row["mA"]),
                                    "receivedQ": int(row.get("receivedQ", row["mQ"])),
                                    "e": int(row.get("e", 0)), "r": int(label[0]),
                                    "b": int(label[1]), "count": int(count)})
        result[basis] = _finish_sample(records, shots, selector, "NumPy / exact Born probabilities")
    return result


def _append_decoder(target, block, qmap, cmap):
    """Connect the validated student operations to the full circuit.

    To avoid the nested-if_test compose bit-remapping issue in Qiskit 2.3,
    create conditions directly on the full circuit's classical bits. Branch global phases
    are omitted because they affect neither outcomes nor joint density matrices.
    """
    for instruction in block.data:
        op = instruction.operation
        qindices = [qmap[block.find_bit(q).index] for q in instruction.qubits]
        cindices = [cmap[block.find_bit(c).index] for c in instruction.clbits]
        if isinstance(op, IfElseOp):
            bit, value = op.condition
            target_bit = target.clbits[cmap[block.find_bit(bit).index]]
            with target.if_test((target_bit, int(value))) as otherwise:
                _append_decoder(target, op.blocks[0], qindices, cindices)
            if len(op.blocks) > 1 and op.blocks[1] is not None:
                with otherwise:
                    _append_decoder(target, op.blocks[1], qindices, cindices)
        elif isinstance(op, Barrier):
            target.barrier(*qindices)
        elif isinstance(op, XGate):
            target.x(qindices[0])
        elif isinstance(op, ZGate):
            target.z(qindices[0])
        elif isinstance(op, IGate):
            target.id(qindices[0])


def build_transfer_circuit(decoder, psi=PHI_PLUS, flip=0, basis=None, measure_output=False):
    """4q/5c dynamic circuit. Store actual outcomes and received records in separate classical bits.

    c0=mQ, c1=mA, c2=receivedQ, c3=r, c4=b.
    flip is fixed for one circuit execution; the calling tool mixes samples according to p.
    """
    decoder_unitaries(decoder)
    if flip not in (0, 1):
        raise ValueError("flip must be 0 or 1.")
    if basis is not None and basis not in BASES:
        raise ValueError("basis must be XX, YY, ZZ, or None.")
    qc = QuantumCircuit(4, 5)
    qc.compose(_teleport_prefix(psi), range(4), inplace=True)
    qc.measure(1, 0)
    qc.measure(2, 1)
    received = expr.bit_not(qc.clbits[0]) if flip else qc.clbits[0]
    qc.store(qc.clbits[2], received)
    _append_decoder(qc, decoder, [3], [2, 1])
    if basis is not None:
        for q, axis in zip((0, 3), basis):
            if axis == "Y":
                qc.sdg(q)
            if axis in ("X", "Y"):
                qc.h(q)
    if measure_output:
        qc.measure(0, 3)
        qc.measure(3, 4)
    return qc


def run_transfer_aer(decoder, psi=PHI_PLUS, p=0, shots=SHOTS, seed=SEED, keep=None):
    """Run actual Aer mid-circuit measurements, classical-record changes, if_test, and final measurements.

    For each setting, draw the number of error trials from Binomial(shots,p), then run the e=0/1 circuits
    the corresponding numbers of times. This has the same sampling law as independent per-trial bit flips.
    """
    shots = _shot_count(shots)
    p = _error_probability(p)
    selector = _selection(keep)
    backend = AerSimulator(max_parallel_threads=1)
    result = {}
    for index, basis in enumerate(BASES):
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), index, 82]))
        errors = int(rng.binomial(shots, p))
        records = []
        for flip, n in ((0, shots-errors), (1, errors)):
            if n == 0:
                continue
            circuit = build_transfer_circuit(decoder, psi, flip, basis, True)
            run_seed = int(seed) + 100 * index + flip
            compiled = transpile(circuit, backend, optimization_level=0, seed_transpiler=run_seed)
            counts = backend.run(compiled, shots=n, seed_simulator=run_seed).result().get_counts()
            for key, count in counts.items():
                bits = key.replace(" ", "").zfill(5)[::-1]
                records.append({"mQ": int(bits[0]), "mA": int(bits[1]),
                                "receivedQ": int(bits[2]), "e": flip,
                                "r": int(bits[3]), "b": int(bits[4]), "count": int(count)})
        result[basis] = _finish_sample(records, shots, selector, "Aer / dynamic circuit")
    return result

In [ ]:
from IPython.display import display, Markdown

CHECK_STATUS = {}
LAB_RESULTS = {}


def run_check(label, check):
    try:
        check()
    except NotImplementedError:
        CHECK_STATUS[label] = "Incomplete"
        print(f"[Incomplete] {label}: Complete the TODO, then rerun the answer cell and this check cell.")
    except Exception as error:
        CHECK_STATUS[label] = "Failed"
        print(f"[Failed] {label}: {type(error).__name__}: {error}")
    else:
        CHECK_STATUS[label] = "Passed"
        print(f"[Passed] {label}")


def run_demo(label, experiment, requires=()):
    LAB_RESULTS.pop(label, None)
    if not RUN_OBSERVATIONS:
        print(f"[Waiting] {label}: Write your prediction and reasoning, set RUN_OBSERVATIONS=True, then run this experiment cell.")
        return None
    missing = [key for key in requires if CHECK_STATUS.get(key) != "Passed"]
    if missing:
        print(f"[Waiting] {label}: First pass these code checks: {', '.join(missing)}")
        return None
    try:
        result = experiment()
    except NotImplementedError:
        print(f"[Waiting] {label}: Complete the preceding TODOs.")
        return None
    except Exception as error:
        print(f"[Experiment error] {label}: {type(error).__name__}: {error}")
        return None
    LAB_RESULTS[label] = result
    return result


def _format(value):
    if isinstance(value, (float, np.floating)):
        return f"{value:.6f}"
    return str(value).replace("|", "\\|").replace("\n", " ")


def show_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(_format(v) for v in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def vector_text(vector):
    return np.array2string(np.asarray(vector), precision=4, suppress_small=True)


def sample_fidelity_table(sample, use_keep=False):
    key = "counts_keep" if use_keep else "counts_all"
    expectation = {basis: correlator(sample[basis][key]) for basis in BASES}
    return expectation, bell_fidelity(expectation)


def assert_scalar_close(value, expected, description):
    array = np.asarray(value)
    assert array.shape == () and np.isrealobj(array), f"{description}: Return one real number."
    assert not isinstance(value, (bool, np.bool_)), f"{description}: A real number is required, not bool."
    assert np.isfinite(array), f"{description}: A finite value is required."
    assert np.isclose(float(array), expected, atol=ATOL, rtol=0), (
        f"{description}: computed {value}, reference {expected}")


def probe_inputs(seed=SEED, count=4):
    rng = np.random.default_rng(seed)
    values = [PHI_PLUS, PHASE_INPUT, np.array([1, 0, 0, 0], dtype=complex),
              np.array([0, 1, 0, 0], dtype=complex)]
    for _ in range(count):
        vector = rng.normal(size=4) + 1j * rng.normal(size=4)
        values.append(vector / np.linalg.norm(vector))
    return values


def probe_densities(seed=SEED, count=6):
    rng = np.random.default_rng(seed)
    values = [density(PHI_PLUS), density(PHI_MINUS), MIX_RHO]
    for _ in range(count):
        matrix = rng.normal(size=(4, 4)) + 1j * rng.normal(size=(4, 4))
        rho = matrix @ matrix.conj().T
        values.append(rho / np.trace(rho))
    return values


LAB_CASES = {"①": density(PHI_PLUS), "②": density(PHI_MINUS), "③": MIX_RHO}
print("Provided tools are ready. State vectors and density matrices use R,B basis order 00,01,10,11.")

In [ ]:
RUN_OBSERVATIONS = False
print("Observation experiments:", "Enabled" if RUN_OBSERVATIONS else "Disabled")

## Problem 1. Teleport One Part of an Entangled State

In this problem, you will <strong>complete Bob's correction according to Alice's measurement outcomes</strong>. After correction, R and B must share the same state as the original R and Q. We will first examine what Alice measures and how her outcomes determine the joint state of R and B.

### The Input State and the Shared Resource

The input R,Q and the shared resource A,B are each prepared in the Bell state $|\Phi^+\rangle$. The pairs are initially independent, so the full state is the tensor product of the two Bell states:

$$
|\Omega_{\mathrm{in}}\rangle
=|\Phi^+\rangle_{RQ}\otimes|\Phi^+\rangle_{AB}.
$$

Here, $\otimes$ combines the two pairs into a description of one overall system. The four qubits have the following roles.

| Qubit in the full circuit | Name | Role |
|---|---|---|
| q0 | R | Entangled with the input Q. No operation acts on it during teleportation. |
| q1 | Q | Alice's input qubit. Its quantum information is to be transferred. |
| q2 | A | Alice's half of the shared Bell pair. |
| q3 | B | Bob's half of the shared Bell pair, which becomes the output. |

We label states with qubit names. For example, $|01\rangle_{RB}$ is the component in which R is 0 and B is 1. In the calculations below, read the entries in the order shown by the subscripts.

### Why Does Alice Apply CX and H?

<strong>Alice's measurement connects the input Q to the shared resource A.</strong> Q is entangled with R, while A is entangled with B. By choosing an appropriate joint measurement on Q and A, Alice can leave R and B in an entangled state after the measurement.

We use a <strong>Bell-basis measurement</strong> on Q and A. The Bell basis consists of these four states:

$$
|\Phi^\pm\rangle=\frac{|00\rangle\pm|11\rangle}{\sqrt2},
\qquad
|\Psi^\pm\rangle=\frac{|01\rangle\pm|10\rangle}{\sqrt2}.
$$

Because the four states are mutually orthogonal, they can serve as a measurement basis for two qubits. Just as Z measurement uses $|0\rangle$ and $|1\rangle$ as its basis, this measurement uses the four Bell states as its basis for Q and A. Each outcome corresponds to one Bell state.

The circuit <strong>maps the four Bell states to different combinations of 0 and 1, then performs ordinary Z-basis measurements</strong>. CX and H implement this mapping.

First apply CX with Q as the control and A as the target. It leaves A unchanged in components where Q is 0 and flips A where Q is 1. Consequently, the components $|00\rangle,|11\rangle$, whose bits were equal, give A=0, while the components $|01\rangle,|10\rangle$, whose bits differed, give A=1.

To see how the mapping also distinguishes relative signs, consider $|\Phi^+\rangle$ and $|\Phi^-\rangle$. CX acts as follows:

$$
\frac{|00\rangle_{QA}\pm|11\rangle_{QA}}{\sqrt2}
\xrightarrow{\mathrm{CX}_{Q\to A}}
\frac{|0\rangle_Q\pm|1\rangle_Q}{\sqrt2}\otimes|0\rangle_A
=|\pm\rangle_Q\otimes|0\rangle_A.
$$

After CX, A is $|0\rangle$ in both cases. The relative sign is now carried by Q being $|+\rangle$ or $|-\rangle$. A Z measurement of Q at this point would give 0 and 1 with equal probability in either case, so it could not distinguish the signs. We therefore apply H to Q, mapping $|+\rangle$ to $|0\rangle$ and $|-\rangle$ to $|1\rangle$, and then measure.

Writing the Q and A outcomes as $m_Q,m_A$, respectively, the transformations of these two Bell-basis states are shown below. Both the state subscripts and measurement records use Q,A order.

| Bell-basis state | After CX(Q, A) | After the subsequent H(Q) | Z-measurement record $(m_Q,m_A)$ |
|---|---|---|---|
| $\lvert\Phi^+\rangle_{QA}$ | $\lvert+\rangle_Q\lvert0\rangle_A$ | $\lvert0\rangle_Q\lvert0\rangle_A$ | $(0,0)$ |
| $\lvert\Phi^-\rangle_{QA}$ | $\lvert-\rangle_Q\lvert0\rangle_A$ | $\lvert1\rangle_Q\lvert0\rangle_A$ | $(1,0)$ |

This table shows how the circuit acts on each of two Bell-basis states. It does not mean that Q and A already occupy one of those states before measurement in the actual four-qubit experiment. Applying the same operations to $|\Psi^+\rangle$ and $|\Psi^-\rangle$ distinguishes the remaining two bit combinations.

Alice's circuit therefore proceeds as <strong>CX(Q, A) → H(Q) → Z measurements of Q and A</strong>. The Q outcome $m_Q$ is stored in classical bit c0, and the A outcome $m_A$ in c1. Bob receives these two bits and chooses his correction.

### From Measurement Outcomes to Bob's Correction

<strong>Knowing Alice's measurement outcome lets us calculate the corresponding joint state of R and B.</strong> We call each case conditioned on a measurement outcome a “measurement branch.” One run produces one of the four outcomes, and the correction must match the state of R and B in that branch.

To calculate a branch, take the state immediately before Alice's measurement and retain only terms with the observed values of Q and A. This gives a state vector for R and B. <strong>The sum of the squared magnitudes of its coefficients is the probability of that outcome.</strong> Dividing the vector by the square root of this probability gives the normalized conditional state. The provided branch-inspection tool displays the unnormalized vectors for this calculation.

#### Worked Example: $m_Q=1$, $m_A=0$

Let us calculate this branch directly. Multiplying the two initial Bell states gives four terms, $|0000\rangle$, $|0011\rangle$, $|1100\rangle$, and $|1111\rangle$, each with coefficient $1/2$. Here the four entries are in R,Q,A,B order.

After CX(Q, A), keeping the terms with A=0 gives $(|0000\rangle+|1101\rangle)/2$. The other two terms have A=1 and do not contribute to this outcome. Applying H to Q and retaining terms with Q=1 gives

$$
\frac{|0100\rangle_{RQAB}-|1101\rangle_{RQAB}}{2\sqrt2}
$$

Since Q and A are fixed to 1 and 0, respectively, we can write the remaining R,B state as

$$
|\widetilde\chi_{10}\rangle_{RB}
=\frac{|00\rangle_{RB}-|11\rangle_{RB}}{2\sqrt2}.
$$

The probability of this outcome is

$$
p_{10}
=\left|\frac{1}{2\sqrt2}\right|^2
+\left|-\frac{1}{2\sqrt2}\right|^2
=\frac14
$$

Dividing the vector by $\sqrt{p_{10}}=1/2$ gives the normalized joint state of R and B in this branch:

$$
|\chi_{10}\rangle_{RB}
=\frac{|00\rangle_{RB}-|11\rangle_{RB}}{\sqrt2}
=|\Phi^-\rangle_{RB}
$$

Compared with the target $|\Phi^+\rangle$, the sign of the $|11\rangle$ term is different.

We can undo this difference by <strong>applying Z to B alone</strong>. Z leaves $|0\rangle$ unchanged and reverses the sign of $|1\rangle$, so

$$
(I_R\otimes Z_B)
\frac{|00\rangle_{RB}-|11\rangle_{RB}}{\sqrt2}
=\frac{|00\rangle_{RB}+|11\rangle_{RB}}{\sqrt2}
$$

Here, $I_R$ means that nothing acts on R. <strong>Acting only on B, without touching R, can restore the two-qubit joint state.</strong>

Find the corrections for the other branches in the same way. X exchanges B's $|0\rangle$ and $|1\rangle$ components, while Z changes the sign of its $|1\rangle$ component. Compare the uncorrected state with the target and decide whether to do nothing, apply X or Z, or apply both.

### Express Conditional Corrections in a Circuit

Bob receives two bits recording Alice's measurement outcomes. His correction circuit must therefore <strong>apply the required gates according to the received bit values</strong>. Qiskit's `if_test` checks whether a specified classical bit is 0 or 1 and executes the enclosed operations only when the condition holds. Once you have listed the required correction for each outcome, you can translate that table into conditional operations.

The code below provides Bell-resource preparation, Alice's operations and measurements, and the connections between the circuit components. You complete the TODO in `bob_decoder()`. It must return `QuantumCircuit(1, 2)`, whose local indices mean the following.

| Item inside the function | Meaning |
|---|---|
| q0 | Bob's B, connected to q3 of the full circuit. |
| c0 | Q's measurement outcome $m_Q$ |
| c1 | A's measurement outcome $m_A$ |

The function receives neither the input-preparation circuit nor the input state's coefficients. Bob must choose his correction using only the two measurement outcomes.

### Exercises

#### (a) Complete the Correction Table for All Four Outcomes
For the Bell input $|\Phi^+\rangle_{RQ}$, complete the table below. Give the normalized R,B state in each branch, the probability of that branch, and the correction on B. If you apply two gates, list them in <strong>the order in which the circuit executes them</strong>.

| $m_Q$ | $m_A$ | R,B state before correction | Probability | Correction on B |
|---:|---:|---|---|---|
| 0 | 0 | … | … | … |
| 0 | 1 | … | … | … |
| 1 | 0 | $(\lvert00\rangle_{RB}-\lvert11\rangle_{RB})/\sqrt2$ | $1/4$ | Z |
| 1 | 1 | … | … | … |

First predict the results from the action of the gates, then run the provided branch-inspection cell and compare. You do not need to submit a long expansion of the full four-qubit state. For each row, explain which difference is undone by which operation.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

<strong>Provided Experiment — Inspect the Uncorrected Branches</strong>

Write your predictions in the table above before running this cell. The four vector entries use the R,B basis order 00,01,10,11.

In [ ]:
def inspect_uncorrected():
    rows = uncorrected_branches()
    show_table(["m_Q", "m_A", "Unnormalized RB vector", "Branch probability", "Normalized RB vector"],
        [[r["mQ"], r["mA"], vector_text(r["raw_state"]), r["probability"], vector_text(r["state"])] for r in rows])
    return rows

_ = run_demo("1a", inspect_uncorrected)

#### (b) Write Bob's Correction Circuit
Use your completed table to implement `bob_decoder()`. Keep the provided circuit construction and return format, and add conditional corrections in the TODO region.

Use X, Z, and `if_test` acting on B. Do not act on any other qubit, including R, reprepare the input state, or retain only selected measurement outcomes. Your circuit must <strong>successfully teleport the state for all four of Alice's measurement outcomes</strong>.

Each `if_test` condition must <strong>compare one classical bit of this circuit with 0 or 1</strong>. To check multiple bits together, nest single-bit conditions. The execution tools in this lab do not support comparing an entire classical register such as `qc.cregs[0]` with an integer, or using a classical logical expression as the condition.

<strong>`if_test` Syntax Example</strong>

The following separate circuit illustrates the syntax for a conditional operation.

```python
example = QuantumCircuit(1, 1)

with example.if_test((example.clbits[0], 0)):
    example.h(0)
```

`example.clbits[0]` is the classical bit to check, and the following `0` is the comparison value. In this example, the indented H operation executes only when that bit is 0. Otherwise, it is skipped.

For the TODO below, use your correction table to decide which classical bit to compare with which value, and which X or Z operation to place inside each condition.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

In [ ]:
def bob_decoder():
    qc = QuantumCircuit(1, 2)
    # TODO BEGIN
    # q0=B, c0=m_Q, c1=m_A. Write X/Z corrections for the two measurement outcomes.
    raise NotImplementedError("bob_decoder")
    # TODO END
    return qc

In [ ]:
def check_bob_decoder():
    decoder = bob_decoder()
    for index, psi in enumerate(probe_inputs()):
        for row in corrected_branches(decoder, psi):
            score = fidelity(row["rho"], psi)
            assert_scalar_close(score, 1.0,
                f"Input {index}, joint fidelity for (m_Q,m_A)=({row['mQ']},{row['mA']})")

run_check("1b · Bob correction", check_bob_decoder)

#### (c) Check a Different Input Using the Same Correction
Connect your correction circuit to the provided experiment cell. It displays each measurement outcome's probability and how closely the corrected R,B state matches its target. This agreement is called <strong>fidelity</strong>, a number between 0 and 1. A fidelity of 1 means that the two states represent the same physical state. Use the provided calculation tools here; you will construct a verification method yourself in the next problem.

First inspect all four branches of the Bell input. Then change only the input R,Q to the following state and run the experiment:

$$
|\Psi_{\mathrm{in}}\rangle_{RQ}
=\frac{|00\rangle_{RQ}+i|11\rangle_{RQ}}{\sqrt2}.
$$

This input has a different relative phase between its two components. Keep the Bell resource A,B and Alice's circuit unchanged. <strong>Do not modify Bob's correction function either.</strong> The target is now for R and B to share $(|00\rangle_{RB}+i|11\rangle_{RB})/\sqrt2$.

For both inputs, record the probabilities of the four outcomes and the corrected fidelities. Then connect your results to the correction rules in (a) and explain why the correction function does not need to know the input state.

States that differ only by a common global phase are treated as the same physical state. For example, multiplying the entire state vector by $-1$ does not mean teleportation failed. By contrast, relative phases between components, such as $i$ in this input, must be preserved.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

<strong>Provided Experiment — Check the Transfer of Two Inputs</strong>

Fidelity is calculated from the exact conditional states. The accompanying Aer Z-measurement counts are measurement samples; their distribution alone is not a test of successful teleportation.

In [ ]:
def inspect_corrected_transfer():
    decoder = bob_decoder()
    print("Your Bob correction circuit:")
    print(decoder.draw(output="text"))
    results = {}
    for index, (name, psi) in enumerate((("Bell input", PHI_PLUS), ("Relative-phase i input", PHASE_INPUT))):
        print(name)
        rows = corrected_branches(decoder, psi)
        show_table(["m_Q", "m_A", "Probability", "Corrected RB vector", "F with the correct target"],
            [[r["mQ"], r["mA"], r["probability"], vector_text(r["state"]), fidelity(r["rho"], psi)] for r in rows])
        sample = run_transfer_aer(decoder, psi=psi, keep="all", seed=SEED+index)
        print("Aer joint Z-measurement counts (R,B order):", sample["ZZ"]["counts_all"])
        results[name] = {"branches": rows, "sample": sample}
    return results

_ = run_demo("1c", inspect_corrected_transfer, requires=("1b · Bob correction",))

## Problem 2. Build a Check for Successful Teleportation

In the previous problem, you corrected the output according to Alice's measurement outcomes and checked it with provided tools. Now you will examine <strong>what must be measured to determine whether teleportation succeeded</strong>, and write a function that calculates joint fidelity from measurement results.

The target state in this problem is $|\Phi^+\rangle_{RB}$. If you switched to the relative-phase input at the end of the previous problem, reset the R,Q input to $|\Phi^+\rangle_{RQ}$.

### Do Matching Single-Qubit Statistics Prove Successful Teleportation?

Suppose someone reports the following after checking a teleportation output:

> I measured R and B separately in the Z basis. Each qubit gives 0 and 1 with equal probability, so the target Bell state was successfully transmitted.

This report checks <strong>how often each qubit produces each outcome</strong>, but not <strong>how the two outcomes are related within the same trial</strong>. Compare these two cases. The outcome strings are in R,B order.

| Paired outcomes from the same trial | R alone | B alone | Relationship |
|---|---|---|---|
| 00 and 11, each with probability $1/2$ | 0 and 1 equally often | 0 and 1 equally often | Always equal |
| 01 and 10, each with probability $1/2$ | 0 and 1 equally often | 0 and 1 equally often | Always different |

Counting 0s and 1s separately for each qubit makes the two cases look identical. Pairing the outcomes from the same trial reveals the difference. To check the joint state, we must also examine such <strong>correlations</strong>.

The qubits do not have to be measured together in one location. Even when R and B are measured separately, <strong>recording the two outcomes from the same trial as a pair</strong> lets us determine their correlation. Conversely, shuffling the trial order in the R and B outcome lists destroys the information about how the original outcomes were paired.

### Even Paired Z Measurements Are Not Enough

<strong>Knowing the joint frequencies of Z-measurement outcomes still does not fully verify the target Bell state.</strong> To see why, compare these two preparation procedures.

- <strong>Bell state:</strong> Prepare $|\Phi^+\rangle=(|00\rangle+|11\rangle)/\sqrt2$ in every trial.
- <strong>Mixed state:</strong> In each trial, with probability $1/2$ prepare $|00\rangle$, and with the remaining probability $1/2$ prepare $|11\rangle$.

In the first case, two components occur in a superposition within one state. In the second, different states are prepared probabilistically. The second, mixed state cannot be represented by a single state vector, so we need a broader representation to describe both cases.

We use a <strong>density matrix</strong> to describe pure and mixed states together. The density matrix of a pure state $|\psi\rangle$ is $|\psi\rangle\langle\psi|$: the product of the state column vector and its conjugate-transpose row vector. For a probabilistic preparation of several states, multiply each state's density matrix by its preparation probability and add them.

Using this definition, the two states can be written explicitly below. The two-qubit basis is <strong>$|00\rangle,|01\rangle,|10\rangle,|11\rangle$</strong>, in R,B order. Read both the rows and columns of the matrices in this order.

$$
\rho_{\mathrm{Bell}}
=|\Phi^+\rangle\langle\Phi^+|
=\frac12
\begin{pmatrix}
1&0&0&1\\
0&0&0&0\\
0&0&0&0\\
1&0&0&1
\end{pmatrix}.
$$

$$
\rho_{\mathrm{mix}}
=\frac12|00\rangle\langle00|
+\frac12|11\rangle\langle11|
=\frac12
\begin{pmatrix}
1&0&0&0\\
0&0&0&0\\
0&0&0&0\\
0&0&0&1
\end{pmatrix}.
$$

Below, $\rho_{RB}$ denotes the joint R,B state in this representation. <strong>Z-measurement probabilities are the diagonal entries of the density matrix.</strong> Writing the outcomes of R and B as $r,b$, respectively, the probability of obtaining $rb$ is

$$
P(rb)=\langle rb|\rho_{RB}|rb\rangle
$$

Here, $|rb\rangle$ is the column vector with 1 in the position corresponding to that outcome and 0 elsewhere. Multiplying by it on the right selects that column of the matrix; multiplying by $\langle rb|$ on the left selects the row with the same index. Thus the expression selects a diagonal entry, where the row and column indices agree.

Both matrices have the same diagonal entries, $1/2,0,0,1/2$. They differ in the <strong>off-diagonal entries at row 00, column 11 and row 11, column 00</strong>. These entries are each $1/2$ in the Bell state and 0 in the mixed state.

The difference comes from how the states are prepared. For the Bell density matrix, $|00\rangle$ and $|11\rangle$ occur in the same state vector, producing the cross-component products $|00\rangle\langle11|$ and $|11\rangle\langle00|$. These <strong>cross terms</strong> depend on the relative phase and carry the superposition's relative-phase information. For the mixed state, we instead add the density matrices of $|00\rangle$ and $|11\rangle$ with their preparation probabilities, so there are no cross terms between those two states.

Z measurement reads the identical diagonal entries of the two matrices. In both cases, 00 and 11 each occur with probability $1/2$, while 01 and 10 never occur. <strong>Even confirming that the two outcomes always agree cannot distinguish these states.</strong>

Looking at R alone or B alone also gives identical results. A qubit's <strong>reduced state</strong> describes only its own measurement statistics, excluding the other qubit's information from the joint description. In this example, the reduced state of each of R and B is $I/2$ for both preparations. Here $I$ is the 2×2 identity matrix. A qubit in $I/2$ gives the two outcomes with equal probability in the X, Y, or Z basis. Thus, <strong>changing the basis still cannot distinguish the two preparations if outcomes are counted separately for each qubit</strong>.

We have identified two limitations. Single-qubit statistics miss the relationship between the outcomes; pairing the outcomes but using only the Z basis misses the difference between these density matrices. Changing the measurement basis changes how probabilities are read, so differences invisible to Z measurements may appear in joint results in another basis.

We will therefore <strong>change the measurement basis and compare the correlations between paired outcomes from the same trial</strong>. In this problem, we use three settings: measure both qubits in X, both in Y, or both in Z.

### Calculate Correlations from Paired Outcomes

Write the bits obtained by measuring R and B during verification as $r,b$, respectively. These differ from Alice's correction bits $m_Q,m_A$ in the previous problem. <strong>$m_Q,m_A$ determine the correction during teleportation; $r,b$ test the output after correction.</strong>

<strong>Multiply the two observed values from the same trial and average those products over trials to quantify their correlation.</strong> X, Y, and Z measurements have observed values $+1$ or $-1$, which the circuit records as bits 0 and 1, respectively. For this calculation, interpret bit 0 as $+1$ and bit 1 as $-1$.

| Verification outcomes $rb$ | Observed values | Product |
|---|---|---:|
| 00 | $+1, +1$ | $+1$ |
| 01 | $+1, -1$ | $-1$ |
| 10 | $-1, +1$ | $-1$ |
| 11 | $-1, -1$ | $+1$ |

The product is $+1$ when the bits agree and $-1$ when they differ. Subtracting the probability of different outcomes from that of equal outcomes therefore gives the average product, or expectation value. If $P$ is X, Y, or Z and both qubits are measured in $P$, then

$$
\langle PP\rangle
=P(r=b)-P(r\ne b)
$$

Here, $PP$ abbreviates $P_R\otimes P_B$, acting on R and B respectively. For example, $XX$ is the product of the outcomes obtained by measuring X on each qubit.

In an experiment, we estimate expectation values using counts rather than exact probabilities. Let $N_{00},N_{01},N_{10},N_{11}$ be the counts for the four outcomes <strong>in one chosen measurement setting</strong>. Their sum, $N$, is the total number of trials for that setting. Weighting equal-outcome counts by $+1$ and different-outcome counts by $-1$, then dividing by $N$, gives

$$
\widehat{\langle PP\rangle}
=\frac{N_{00}+N_{11}-N_{01}-N_{10}}{N}
$$

The hat denotes an <strong>estimate</strong> obtained from finitely many trials.

To practice the calculation, consider this separate example record for one measurement setting.

| Verification outcomes $rb$ | 00 | 01 | 10 | 11 |
|---|---:|---:|---:|---:|
| Count | 40 | 10 | 20 | 30 |

There are 100 trials: $40+30=70$ with equal outcomes and $10+20=30$ with different outcomes. The estimated correlation is therefore

$$
\widehat{\langle PP\rangle}
=\frac{40+30-10-20}{100}
=\frac{70-30}{100}
=0.4
$$

Apply this calculation separately to the records for $XX,YY,ZZ$. Do not pool the counts from the three settings; use each setting's own record and number of trials to calculate its average.

### How Do We Implement X and Y Measurements?

We have calculated correlations from records in a chosen basis. Now let us see how a circuit measures in X, Y, or Z. The circuit's default measurement is in the Z basis. To measure in another basis, <strong>first map that basis's two states to $|0\rangle$ and $|1\rangle$</strong>. The 0 and 1 recorded by the subsequent Z measurement can then be read as outcomes in the original basis.

Applying H to the X-basis states $|+\rangle,|-\rangle$ gives $|0\rangle,|1\rangle$, respectively. Thus H followed by Z measurement implements X measurement.

The Y-basis states are $|+i\rangle=(|0\rangle+i|1\rangle)/\sqrt2$ and $|-i\rangle=(|0\rangle-i|1\rangle)/\sqrt2$. The gate $S^\dagger$ multiplies the $|1\rangle$ component by $-i$, mapping these states to $|+\rangle$ and $|-\rangle$, respectively. Applying H next gives

$$
\begin{aligned}
|+i\rangle&\xrightarrow{S^\dagger}|+\rangle\xrightarrow{H}|0\rangle,\\
|-i\rangle&\xrightarrow{S^\dagger}|-\rangle\xrightarrow{H}|1\rangle.
\end{aligned}
$$

Y measurement is therefore implemented by $S^\dagger$, then H, then Z measurement. To measure both qubits in the same basis, <strong>choose one setting below and apply the listed operations to each of R and B</strong>.

| Measurement setting | Operations on each of R and B, in execution order |
|---|---|
| $ZZ$ | Z measurement |
| $XX$ | H → Z measurement |
| $YY$ | $S^\dagger$ → H → Z measurement |

The basis-change and measurement code is provided. <strong>Prepare the state anew and rerun teleportation for every measurement setting and every trial.</strong> In one trial, measure both qubits in the chosen basis and record their results. Testing the next basis starts again from state preparation; it does not mean applying X, Y, and Z measurements successively to the same output.

These operations belong to verification after teleportation. Rotating or measuring R for verification is separate from the earlier requirement to leave R untouched during the transfer itself.

### Obtain Joint Fidelity from Three Correlations

We now use these correlations to calculate <strong>how closely the output matches the target Bell state</strong>. For target $|\Phi^+\rangle$ and actual output $\rho_{RB}$, the joint fidelity is

$$
F_{\Phi^+}
=\langle\Phi^+|\rho_{RB}|\Phi^+\rangle
$$

It lies between 0 and 1, and an exact value of 1 means that the output equals the target state. For a pure output $|\psi\rangle$, the expression becomes $|\langle\Phi^+|\psi\rangle|^2$. Density matrices let us apply the same check to mixed states as well.

To obtain this value from measurements, consider the <strong>projector</strong> that selects the target-state component:

$$
\Pi_{\Phi^+}=|\Phi^+\rangle\langle\Phi^+|.
$$

It leaves the target state unchanged and maps components orthogonal to the target to zero. Applying it to a pure output $|\psi\rangle$ leaves

$$
\Pi_{\Phi^+}|\psi\rangle
=|\Phi^+\rangle\langle\Phi^+|\psi\rangle
$$

As in the previous problem, the squared norm of this vector gives a probability:

$$
\bigl\|\Pi_{\Phi^+}|\psi\rangle\bigr\|^2
=|\langle\Phi^+|\psi\rangle|^2
=F_{\Phi^+}
$$

Thus fidelity is the probability of projection onto the target state. The same relation holds for a mixed state after weighting each prepared component by its probability. This pass probability is the expectation value of $\Pi_{\Phi^+}$.

We use the following decomposition in this lab:

$$
\Pi_{\Phi^+}
=\frac14\left(
I_R\otimes I_B
+X_R\otimes X_B
-Y_R\otimes Y_B
+Z_R\otimes Z_B
\right).
$$

The expectation value of a sum is the sum of the expectation values with the same coefficients. The expectation of $I_R\otimes I_B$ is 1; the remaining terms are the previously measured $\langle XX\rangle$, $\langle YY\rangle$, and $\langle ZZ\rangle$. Use this relation to implement a fidelity function. Bell-state verification using the same three bases also appears in [IBM Quantum's measurement example](https://quantum.cloud.ibm.com/docs/en/tutorials/long-range-entanglement).

This calculation gives <strong>fidelity with one specified target state</strong>. Three expectation values do not fully determine an arbitrary two-qubit state. A different target also requires a check appropriate to that target. In particular, do not reuse this formula for the relative-phase input from the previous problem and expect a value of 1.

### Exercises

#### (a) Identify Outputs That the Original Checks Miss
The three provided outputs are listed below. All describe R and B.

| Output | State |
|---|---|
| ① | Target Bell state $\lvert\Phi^+\rangle=(\lvert00\rangle+\lvert11\rangle)/\sqrt2$ |
| ② | Bell state with the opposite relative sign, $\lvert\Phi^-\rangle=(\lvert00\rangle-\lvert11\rangle)/\sqrt2$ |
| ③ | A mixture preparing $\lvert00\rangle$ and $\lvert11\rangle$, each with probability $1/2$ |

The original report checks only <strong>whether each qubit gives 0 and 1 equally often in Z measurement</strong>. An extended check also tests <strong>whether joint Z measurement gives 00 and 11 equally often</strong>.

Before running code, predict which of the three outputs each check will accept. Then use the provided cell to compare the individual X·Y·Z statistics and the joint Z distribution. Explain whether these results alone distinguish output ① from the other two.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

In [ ]:
def inspect_local_checks():
    rows = []
    for name, rho in LAB_CASES.items():
        local = local_expectations(rho)
        rows.append([name] + [local[q][axis] for q in ("R", "B") for axis in ("X", "Y", "Z")])
    show_table(["Output", "R:X", "R:Y", "R:Z", "B:X", "B:Y", "B:Z"], rows)
    show_table(["Output", "P(00)", "P(01)", "P(10)", "P(11)"],
        [[name] + list(z_distribution(rho).values()) for name, rho in LAB_CASES.items()])
    print("For each qubit, the probability of +1 (bit 0) is (1+its expectation value)/2.")
    return {name: {"local": local_expectations(rho), "ZZ": z_distribution(rho)} for name, rho in LAB_CASES.items()}

_ = run_demo("2a", inspect_local_checks)

#### (b) Write a Function That Calculates Fidelity from Correlations
Complete the TODO in `bell_fidelity(correlations)`. Its input is a dictionary containing these three expectation values.

| Entry | Meaning |
|---|---|
| `correlations["XX"]` | $\langle XX\rangle$ |
| `correlations["YY"]` | $\langle YY\rangle$ |
| `correlations["ZZ"]` | $\langle ZZ\rangle$ |

Use the projector decomposition above to return one real number: the fidelity with target $|\Phi^+\rangle$. Calculate from the three supplied expectations, rather than assigning a value based on the output's label.

The circuits that change measurement bases and the tools that extract correlations from counts are provided. Your function <strong>combines the three correlations into one check value for the target state</strong>.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

In [ ]:
def bell_fidelity(correlations):
    # TODO BEGIN
    # Return fidelity with target Phi+ from the XX, YY, ZZ expectations.
    raise NotImplementedError("bell_fidelity")
    # TODO END

In [ ]:
def check_bell_fidelity():
    for index, rho in enumerate(probe_densities()):
        observed = bell_fidelity(correlations(rho))
        expected = fidelity(rho, PHI_PLUS)
        assert_scalar_close(observed, expected, f"Bell fidelity for test state {index}")

run_check("2b · Joint fidelity", check_bell_fidelity)

#### (c) Compare the Two Calculations and Apply the Check to Teleportation
First feed the exact expectations for the three outputs into your function and complete the table. The directly calculated fidelity is obtained by the provided tool using $\langle\Phi^+|\rho_{RB}|\Phi^+\rangle$.

| Output | $\langle XX\rangle$ | $\langle YY\rangle$ | $\langle ZZ\rangle$ | Fidelity from your function | Fidelity directly from the density matrix |
|---|---|---|---|---|---|
| ① | … | … | … | … | … |
| ② | … | … | … | … | … |
| ③ | … | … | … | … | … |

Check that the two calculations agree within numerical tolerance, and explain how the new check distinguishes outputs that the earlier checks could not. When interpreting output ②, also consider <strong>whether “different from the target state” means the same thing as “not entangled”</strong>.

Next apply this check to the teleportation circuit you wrote in Problem 1. Use input $|\Phi^+\rangle_{RQ}$ and test R and B after Bob's correction. The provided tools calculate exact expectations and measurement samples for the output including all four of Alice's outcomes.

Check the fidelity calculated from exact expectations first, then compare it with the estimate from 4096 measurements per setting. Correctness of your function is tested using <strong>calculations with exact expectation values</strong>. Measurement samples are used to explore the relationship between theoretical values and measurement results.

When the product of the two outcomes varies between trials, as in X·Y measurements of output ③, the sample estimate can fluctuate around the theoretical value. For the ideal target Bell state, by contrast, each product outcome for $XX,YY,ZZ$ is fixed, so the estimated fidelity from this check can be exactly 1 even with finitely many trials. Distinguish these cases when explaining your results.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

In [ ]:
def compare_fidelity_methods():
    rows, results = [], {}
    for index, (name, rho) in enumerate(LAB_CASES.items()):
        exact = correlations(rho)
        sampled = {basis: correlator(sample_state(rho, basis, seed=SEED+100+10*index+j))
                   for j, basis in enumerate(BASES)}
        score = bell_fidelity(exact)
        rows.append([name] + list(exact.values()) + [score, fidelity(rho), bell_fidelity(sampled)])
        results[name] = {"exact": exact, "F": score, "sampled": sampled}
    show_table(["Output", "XX", "YY", "ZZ", "Student-function F", "Density-matrix F", "4096-shot estimated F"], rows)
    print("Samples for the three example states were drawn from exact Born probabilities using NumPy.")
    return results

_ = run_demo("2c_cases", compare_fidelity_methods, requires=("2b · Joint fidelity",))


def verify_actual_transfer():
    decoder = bob_decoder()
    rho = weighted_density(corrected_branches(decoder))
    sample = run_transfer_aer(decoder, keep="all", seed=SEED+200)
    sampled_E, sampled_F = sample_fidelity_table(sample)
    show_table(["Calculation", "XX", "YY", "ZZ", "Joint F"], [
        ["Exact output"] + list(correlations(rho).values()) + [bell_fidelity(correlations(rho))],
        ["4096-shot sample from the actual Aer dynamic circuit"] + list(sampled_E.values()) + [sampled_F],
    ])
    return {"rho": rho, "sample": sample, "F": bell_fidelity(correlations(rho))}

_ = run_demo("2c", verify_actual_transfer, requires=("1b · Bob correction", "2b · Joint fidelity"))

## Problem 3. Review a Report Based on Selected Outcomes

In the previous problem, you chose measurements for checking the teleportation output. Now we examine <strong>which trials were included in that check</strong>. Even for the same circuit, selecting only some outcomes can change the reported fidelity.

The input remains $|\Phi^+\rangle_{RQ}$, and the target output is $|\Phi^+\rangle_{RB}$. Here, “overall teleportation performance” means <strong>performance including every trial for this input</strong>. It does not mean an average over all possible input states.

### What If Fidelity Is 1 Only in the Selected Trials?

We now use a circuit that <strong>applies Bob's X correction correctly but omits the Z correction</strong>. Bell-resource preparation and Alice's operations and measurements are unchanged. Run the provided variant as it is and analyze the output with the missing correction.

Someone runs this circuit, retains only the trials in which Alice obtained $m_Q=0$, and reports:

> I calculated the joint fidelity of R and B in the retained trials and obtained 1. Therefore, this circuit successfully teleports the state.

To assess this report, distinguish <strong>the state of the retained trials</strong> from <strong>the state of all trials</strong>. Excluding some trials and recalculating can raise the fidelity of those that remain. It does not correct the outputs of the excluded trials.

Retaining only trials whose measurement outcomes satisfy a specified condition is called <strong>postselection</strong>. Reporting both the selection condition and the fraction retained makes performance under that condition meaningful. It must not be interpreted as the performance obtained across all trials.

### What State Describes All Four Measurement Branches Together?

Write Alice's paired outcomes as $m=(m_Q,m_A)$. Let $p_m$ be the probability of an outcome and $\rho_m$ the corresponding joint R,B state. <strong>$\rho_m$ is the normalized state after Bob's operations in the circuit being analyzed.</strong> In this variant, it is the state after the X correction only.

One trial produces one of the four outcomes. Since they cover all possibilities, their probabilities sum to $\sum_m p_m=1$. The output of all trials pooled without distinguishing outcomes is the probability-weighted average of the branch density matrices:

$$
\rho_{\mathrm{all}}=\sum_m p_m\rho_m.
$$

This is the same way we represented a mixture of different preparations in Problem 2. We are combining outputs from different trials, not adding state vectors to form a coherent superposition.

If each branch fidelity is calculated relative to the same target $|\Phi^+\rangle$ as

$$
F_m=\langle\Phi^+|\rho_m|\Phi^+\rangle
$$

then the overall output fidelity is

$$
\begin{aligned}
F_{\mathrm{all}}
&=\langle\Phi^+|\rho_{\mathrm{all}}|\Phi^+\rangle\\
&=\sum_m p_m\langle\Phi^+|\rho_m|\Phi^+\rangle\\
&=\sum_m p_mF_m
\end{aligned}
$$

Substituting $\rho_{\mathrm{all}}=\sum_m p_m\rho_m$ in the first line and expanding gives <strong>the sum of each branch's fidelity multiplied by its probability</strong>. This works here because every branch is compared with the same pure target state.

### What Changes When Only Some Trials Are Retained?

Let $S$ be the set of retained outcomes. Our selection condition is $m_Q=0$, so

$$
S=\{(0,0),(0,1)\}
$$

The probability that a trial satisfies this condition—the <strong>retained fraction</strong>—is

$$
p_{\mathrm{keep}}=\sum_{m\in S}p_m
$$

In measurement records, estimate it by dividing the number of retained trials by the total number of trials.

Now treat the retained trials as their own ensemble. The fraction contributed by each branch within this ensemble differs from its unconditional probability $p_m$. Divide each probability by $p_{\mathrm{keep}}$ so that <strong>the probabilities within the selected ensemble again sum to 1</strong>. Its output state is therefore

$$
\rho_{\mathrm{keep}}
=\sum_{m\in S}\frac{p_m}{p_{\mathrm{keep}}}\rho_m
=\frac{\sum_{m\in S}p_m\rho_m}{p_{\mathrm{keep}}}
$$

We consider selections with nonzero probability, $p_{\mathrm{keep}}>0$. This normalization of a state conditioned on measurement outcomes is also described in [IBM Quantum's explanation of conditional states](https://quantum.cloud.ibm.com/learning/en/courses/general-formulation-of-quantum-information/general-measurements/formulations-of-measurements).

The <strong>fidelity of the selected trials</strong> with the same target uses the same normalized weights:

$$
F_{\mathrm{keep}}
=\frac{\sum_{m\in S}p_mF_m}{p_{\mathrm{keep}}}.
$$

$F_{\mathrm{keep}}$ describes how closely the retained output matches the target. $p_{\mathrm{keep}}$ describes how often that output is retained among all trials. <strong>The quality of the selected output and the fraction of trials that produce it are different pieces of information.</strong>

### Worked Example: Retained and Discarded Trials

To practice the calculation, consider an example separate from our teleportation circuit. Suppose an experiment has the following two cases, with normalized output states $\rho_1,\rho_2$. Both fidelities are calculated relative to the same pure target.

| Case | Output state | Probability | Fidelity | Selection |
|---|---|---:|---:|---|
| First | $\rho_1$ | 0.8 | 1 | Retain |
| Second | $\rho_2$ | 0.2 | 0.5 | Discard |

Summarize this record in two ways. <strong>Including all trials</strong> averages the two outputs:

$$
\rho_{\mathrm{all}}=0.8\rho_1+0.2\rho_2
$$

<strong>Retaining only the first case</strong> gives a retained fraction of $p_{\mathrm{keep}}=0.8$. Its contribution relative to all trials is $0.8\rho_1$, but it makes up the entire selected ensemble. Divide by the retained probability 0.8 to normalize:

$$
\rho_{\mathrm{keep}}=\frac{0.8\rho_1}{0.8}=\rho_1
$$

The denominator 0.8 is the <strong>probability of satisfying the selection condition</strong>, not the agreement with the target. Using the same denominator for the selected fidelity gives

$$
F_{\mathrm{keep}}=\frac{0.8\times1}{0.8}=1
$$

By contrast, the fidelity including every trial is

$$
F_{\mathrm{all}}=0.8\times1+0.2\times0.5=0.9
$$

A selected fidelity of 1 and an overall fidelity of 0.9 are consistent. <strong>They describe different ensembles.</strong>

In this example, $p_{\mathrm{keep}}F_{\mathrm{keep}}=0.8$ is the retained trials' contribution to the overall fidelity. Adding the discarded trials' contribution, $0.2\times0.5=0.1$, gives the total 0.9. Excluding trials from an analysis does not remove their target-state components from the calculation over all trials.

When both retained and discarded trials exist, write the fidelity of the discarded ensemble as $F_{\mathrm{drop}}$. Then

$$
F_{\mathrm{all}}
=p_{\mathrm{keep}}F_{\mathrm{keep}}
+(1-p_{\mathrm{keep}})F_{\mathrm{drop}}
$$

If all trials are retained, $p_{\mathrm{keep}}=1$ and $F_{\mathrm{all}}=F_{\mathrm{keep}}$.

### How Can We Reuse the Measurement Check?

You can also use `bell_fidelity()` from Problem 2 on the selected trials. Calculate their $XX,YY,ZZ$ correlations first, then pass those three values to the function.

Apply <strong>the same selection condition in all three measurement settings</strong>. Here, retain trials satisfying $m_Q=0$ in each setting. This ensures that all three correlations refer to the same conditional output state $\rho_{\mathrm{keep}}$.

Accordingly, summarize the measurement records in two ways. For overall performance, use all trials from that setting. For selected performance, retain only trials satisfying the condition, then recount 00·01·10·11 within that subset. The denominator of each correlation must be <strong>the number of selected trials in that measurement setting</strong>. Dividing by the original total would not give an average within the selected ensemble. Separately report the retained fraction as “selected trials / total trials.”

Selection uses Alice's measurement bit $m_Q$. Looking at the verification outcomes $r,b$ and retaining only favorable ones is not part of this selection rule. The provided tools select records and compute correlations; connect the `bell_fidelity()` function you already wrote without changing it.

### Exercises

#### (a) Inspect All Four Branches of the Incomplete Correction
Set both the input and the shared resource to $|\Phi^+\rangle$, and analyze the provided circuit that omits Z correction. Predict the state remaining on R and B <strong>after the X correction</strong> in each branch, and complete the table with normalized states.

| $m_Q$ | $m_A$ | R,B state after X correction | Branch probability $p_m$ | Fidelity with the target $F_m$ |
|---:|---:|---|---|---|
| 0 | 0 | … | … | … |
| 0 | 1 | … | … | … |
| 1 | 0 | … | … | … |
| 1 | 1 | … | … | … |

First predict the results using your correction table from Problem 1, then run the branch-inspection cell and compare. You can verify $F_m$ by passing each branch's exact correlations to `bell_fidelity()` from Problem 2.

Use the table to explain which states are selected when retaining only trials with $m_Q=0$. Identify precisely which trials the fidelity in the original report describes.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

In [ ]:
def inspect_missing_z():
    rows = corrected_branches(missing_z_decoder())
    show_table(["m_Q", "m_A", "RB vector after X correction", "Probability", "Joint F"],
        [[r["mQ"], r["mA"], vector_text(r["state"]), r["probability"],
          bell_fidelity(correlations(r["rho"]))] for r in rows])
    return rows

_ = run_demo("3a", inspect_missing_z, requires=("2b · Joint fidelity",))

#### (b) Calculate the Retained Fraction and the Two Fidelities
Complete `summarize_selection(probabilities, fidelities, keep_mask)`. Its three inputs are sequences arranged in the same order.

| Input | Meaning of each entry |
|---|---|
| `probabilities` | Branch probability $p_m$ relative to all trials |
| `fidelities` | Fidelity $F_m$ of that branch |
| `keep_mask` | `True` to retain the branch, `False` to discard it |

The branches of this circuit are ordered $(m_Q,m_A)=(0,0),(0,1),(1,0),(1,1)$. Thus the condition selecting $m_Q=0$ is passed as `[True, True, False, False]`.

Return <strong>a tuple in the order `(p_keep, F_keep, F_all)`</strong>. You may assume that the probabilities sum to 1 and that the sum over selected branches is positive. Use the supplied probabilities rather than assuming that every branch is equally likely.

The provided tools directly average the density matrices. Your function obtains the same quantities by probability-weighting the branch fidelities. Check that the two methods agree.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

In [ ]:
def summarize_selection(probabilities, fidelities, keep_mask):
    # TODO BEGIN
    # Use the occurrence probabilities to return (p_keep, F_keep, F_all).
    raise NotImplementedError("summarize_selection")
    # TODO END

In [ ]:
def check_selection_summary():
    examples = [
        ([0.8, 0.2], [1.0, 0.5], [True, False], (0.8, 1.0, 0.9)),
        ([0.1, 0.2, 0.3, 0.4], [0.3, 0.6, 0.1, 0.9],
         [True, False, False, True], (0.5, 0.78, 0.54)),
        ([0.25] * 4, [0.2, 0.7, 0.9, 0.1], [True] * 4, (1.0, 0.475, 0.475)),
        ([0.15, 0.85], [0.1, 0.9], [False, True], (0.85, 0.9, 0.78)),
    ]
    for index, (probabilities, fidelities, mask, expected) in enumerate(examples):
        result = summarize_selection(probabilities, fidelities, mask)
        assert isinstance(result, tuple) and len(result) == 3, "Return a tuple of three real numbers."
        for name, value, reference in zip(("p_keep", "F_keep", "F_all"), result, expected):
            assert_scalar_close(value, reference, f"Example {index}: {name}")

run_check("3b · Selection summary", check_selection_summary)

#### (c) Compare Three Reports
Use your function to compare the three cases below. For normal correction, use `bob_decoder()` from Problem 1. For the omitted-Z cases, use the provided variant without modifying your normal correction function.

| Circuit and selection | Retained fraction $p_{\mathrm{keep}}$ | Selected-trial $F_{\mathrm{keep}}$ | All-trial $F_{\mathrm{all}}$ |
|---|---|---|---|
| Normal correction; retain all trials | … | … | … |
| Omit Z correction; retain all trials | … | … | … |
| Omit Z correction; retain only $m_Q=0$ trials | … | … | … |

First record the values calculated from exact branch probabilities. Then run the provided measurement cell and compare them with the retained fractions and fidelity estimates from 4096 trials per basis. With finite samples, the number of selected trials can differ slightly between bases. Code checks use results calculated from exact probabilities.

Finally, rewrite the original report in two or three sentences. Include <strong>the selection condition, retained fraction, selected-trial fidelity, and all-trial fidelity</strong>. Explain what changed and what remained unchanged when only the selection condition changed for the same circuit.

The teleportation goal in this lab is to restore the input for every one of Alice's measurement outcomes. Use your revised report to judge whether the circuit omitting Z correction meets that goal.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

In [ ]:
def compare_selection_reports():
    normal, missing = bob_decoder(), missing_z_decoder()
    rows_out, theory = [], {}
    for name, decoder, select in [
        ("Normal correction / all", normal, lambda r: True),
        ("Omit Z / all", missing, lambda r: True),
        ("Omit Z / m_Q=0", missing, lambda r: r["mQ"] == 0),
    ]:
        branches = corrected_branches(decoder)
        probabilities = [r["probability"] for r in branches]
        scores = [bell_fidelity(correlations(r["rho"])) for r in branches]
        keep = [select(r) for r in branches]
        summary = summarize_selection(probabilities, scores, keep)
        kept_rho = weighted_density([r for r, k in zip(branches, keep) if k])
        all_rho = weighted_density(branches)
        assert np.allclose(summary[1:], [fidelity(kept_rho), fidelity(all_rho)], atol=ATOL, rtol=0)
        rows_out.append([name] + list(summary))
        theory[name] = {"summary": summary, "kept_rho": kept_rho, "all_rho": all_rho}
    show_table(["Circuit and condition", "Retained fraction", "Selected F", "Overall F"], rows_out)
    normal_sample = run_transfer_aer(normal, keep="all", seed=SEED+300)
    missing_sample = run_transfer_aer(missing, seed=SEED+301)
    normal_F = sample_fidelity_table(normal_sample)[1]
    missing_F = sample_fidelity_table(missing_sample)[1]
    kept_F = sample_fidelity_table(missing_sample, use_keep=True)[1]
    show_table(["4096-shot sample", "Estimated selected F", "Estimated overall F"], [
        ["Normal / all", normal_F, normal_F],
        ["Omit Z / all", missing_F, missing_F],
        ["Omit Z / m_Q=0", kept_F, missing_F],
    ])
    show_table(["Omitted-Z circuit setting", "Total trials", "Selected trials", "Estimated retained fraction"],
        [[basis, sample["total"], sample["kept"], sample["selection_rate"]]
         for basis, sample in missing_sample.items()])
    return {"theory": theory, "normal_sample": normal_sample, "missing_sample": missing_sample}

_ = run_demo("3c", compare_selection_reports,
    requires=("1b · Bob correction", "2b · Joint fidelity", "3b · Selection summary"))

## Problem 4. What If the Measurement Record Is Transmitted Incorrectly?

The previous problem showed how excluding trials can change reported teleportation performance. We now <strong>include every trial but introduce errors in the measurement record sent to Bob</strong>. Calculate how an incorrect record affects the correction, then decide whether changing the correction rule can restore the state.

Do not use the omitted-Z circuit from Problem 3. <strong>Return to the normal circuit with both X and Z corrections.</strong> The input and shared resource are $|\Phi^+\rangle_{RQ}$ and $|\Phi^+\rangle_{AB}$, respectively, and the target output is $|\Phi^+\rangle_{RB}$.

### The Bit Alice Sends and the Bit Bob Receives

Alice's measurement itself is performed correctly. Errors occur while its outcome is being transmitted to Bob. Assume that <strong>only the Z-correction bit $m_Q$ is flipped in transmission with probability $p$</strong>. The X-correction bit $m_A$ is transmitted correctly.

Errors occur independently in each trial, independently of the input state and Alice's measurement outcomes. The quantity $p$ is the probability of a bit flip in one trial and lies between 0 and 1.

Use the following notation to distinguish the original outcome from the received record.

| Symbol | Meaning |
|---|---|
| $m_Q$ | Alice's actual Q-measurement outcome |
| $\widetilde m_Q$ | The Q-measurement record received by Bob |
| $e$ | 0 if the bit is unchanged in transmission; 1 if it is flipped |

The relationship is

$$
\widetilde m_Q=m_Q\oplus e,
\qquad
\Pr(e=1)=p,\quad \Pr(e=0)=1-p
$$

Here, $\oplus$ is the XOR of two bits. Thus $e=0$ preserves the original bit, while $e=1$ exchanges 0 and 1.

Bob knows <strong>the two received bits $\widetilde m_Q,m_A$ and the error probability $p$</strong>. He does not separately receive Alice's original outcome $m_Q$ or the error indicator $e$. Even if the simulation can calculate those records, only the three stated quantities are available to Bob's correction.

We begin with Bob <strong>trusting the received bits and applying the original correction rule</strong>. In one trial, Alice obtains $m_Q$, and the transmission error indicator $e$ determines the record $\widetilde m_Q$ delivered to Bob. Bob applies X according to the correctly received $m_A$, then applies Z if his received $\widetilde m_Q$ equals 1. His correction is chosen from the received record, not the original outcome. We will consider changing this rule using $p$ in the final subquestion.

### Why Does an Incorrect Bit Cause an Error in the Quantum State?

Correction is needed because the R,B state depends on Alice's actual outcome. If Bob receives a different outcome, he applies <strong>a correction that does not match the state actually present</strong>.

For example, suppose Alice obtains $(m_Q,m_A)=(0,1)$. As established in Problem 1, the uncorrected R,B state is

$$
|\Psi^+\rangle_{RB}
=\frac{|01\rangle_{RB}+|10\rangle_{RB}}{\sqrt2}
$$

Since $m_A=1$, applying X to B gives the target $|\Phi^+\rangle$. Because the original value is $m_Q=0$, Z should not be applied.

But if $m_Q$ is flipped and arrives as $\widetilde m_Q=1$, Bob also applies an unnecessary Z:

$$
|\Psi^+\rangle_{RB}
\xrightarrow{X_B}|\Phi^+\rangle_{RB}
\xrightarrow{Z_B}|\Phi^-\rangle_{RB}.
$$

The resulting state is the target with an extra Z applied.

Conversely, if the original value is $m_Q=1$ but arrives as $\widetilde m_Q=0$, Bob omits a necessary Z correction. The Z effect remaining after the X correction is left undone. <strong>Adding an unnecessary Z and omitting a necessary Z both leave the output one Z operation away from the target.</strong> This follows from $Z^2=I$: applying Z twice restores the state.

We can also verify this algebraically. After the correct X correction, the state differs from the target by $Z_B^{m_Q}$ on B. Here $Z_B^0=I_B$ and $Z_B^1=Z_B$, and we ignore a common global phase. If Bob applies $Z_B^{\widetilde m_Q}$ according to the received record, the residual operation on B is

$$
Z_B^{\widetilde m_Q}Z_B^{m_Q}
=Z_B^{\widetilde m_Q\oplus m_Q}
=Z_B^e
$$

When both exponents are 1, $Z_B^2=I_B$, so the combined exponent can be expressed as XOR. Substituting $\widetilde m_Q=m_Q\oplus e$ makes the original outcome appear twice and cancel, leaving only the error indicator $e$.

With the original correction rule, transmission errors therefore reduce to two cases.

- If the bit is transmitted correctly, the target state is restored.
- If the bit is flipped, the output is the target state with Z applied to B.

Alice still has four possible measurement outcomes. The two cases above are the <strong>two transmission-error cases</strong>, which occur separately from those outcomes. After Bob's correction, the residual difference from the target can be described solely by the error indicator.

### Combine the Two Cases to Calculate the Overall Output

In Problem 3, we averaged the outputs of different measurement branches with their probabilities. Now we average <strong>the outputs for correct transmission and a flipped bit</strong> with their respective probabilities. All trials are included.

Let $\rho_{\mathrm{target}}$ be the joint state that normal teleportation should produce. For our Bell input,

$$
\rho_{\mathrm{target}}=|\Phi^+\rangle\langle\Phi^+|
$$

First consider how to apply the residual Z effect to a density matrix.

Applying a gate $U$ changes a state vector from $|\psi\rangle$ to $U|\psi\rangle$. The density matrix therefore transforms as

$$
|\psi\rangle\langle\psi|
\longmapsto
U|\psi\rangle\langle\psi|U^\dagger
=U\rho U^\dagger
$$

Here, $U^\dagger$ is the conjugate transpose of $U$. The same transformation applies to a mixed state because the gate acts on each component in the same way.

R is unchanged and only B has a residual Z, so $U=I_R\otimes Z_B$. Since Z equals its own conjugate transpose, the output in the flipped-bit case is

$$
\rho_{\mathrm{flip}}
=(I_R\otimes Z_B)\rho_{\mathrm{target}}(I_R\otimes Z_B)
$$

The two cases, their states, and their probabilities are summarized below.

| Transmission error | Record received by Bob | Output after the original correction | Probability |
|---|---|---|---|
| $e=0$ | $\widetilde m_Q=m_Q$ | $\rho_{\mathrm{target}}$ | $1-p$ |
| $e=1$ | $\widetilde m_Q=m_Q\oplus1$ | $\rho_{\mathrm{flip}}$ | $p$ |

This table separates cases for an analyst who knows whether an error occurred. It does not give Bob the additional information $e$. After any of Alice's four outcomes and the original correction, the residual difference is determined by $e$. The overall output is therefore the probability-weighted sum of these two states:

$$
\rho_{\mathrm{out}}(p)
=\Pr(e=0)\rho_{\mathrm{target}}
+\Pr(e=1)\rho_{\mathrm{flip}}
$$

Substituting the two probabilities gives

$$
\rho_{\mathrm{out}}(p)
=(1-p)\rho_{\mathrm{target}}
+p\rho_{\mathrm{flip}}
$$

This means that <strong>the normal output occurs with probability $1-p$, while an output with the residual Z effect occurs with probability $p$</strong>. This is a stochastic phase-flip error model, also used in [IBM Quantum's phase-flip example](https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models).

Apply your earlier check to $\rho_{\mathrm{out}}(p)$ to calculate joint fidelity for each $p$. The output also depends on Bob's correction rule. Remember that the expression derived here assumes <strong>the original correction rule is applied directly to the received bits</strong>.

### Distinguish the Original Correction's Performance from Recoverability

Low fidelity under the original correction does not by itself prove that recovery remains impossible after changing the correction rule. Finally, we will consider whether Bob can use his knowledge of $p$ to change that rule.

First ask <strong>whether Alice's original measurement outcome can be inferred from the received record</strong>. If one received record is consistent with several cases, also check whether those cases require the same correction.

Recall that gates such as X and Z are reversible. Applying the same reversible operation to two distinct states cannot turn both into one identical state: reversing that operation would then have to turn a single state into two different states.

This property lets us determine <strong>whether Bob can restore every possible case using one correction rule for a given received record</strong>. Analyze $p=1$ and $p=1/2$ separately in the following subquestion.

### Exercises

#### (a) Calculate the Output State with Transmission Errors
Complete the TODO in `noisy_transfer_density(rho_target, p)`.

| Input | Meaning |
|---|---|
| `rho_target` | The joint R,B density matrix that normal teleportation should produce; a 4×4 matrix. |
| `p` | Probability that the Z-correction bit is flipped, with $0\le p\le1$. |

Rows and columns use the basis order $|00\rangle_{RB},|01\rangle_{RB},|10\rangle_{RB},|11\rangle_{RB}$. For the 4×4 matrix $I_R\otimes Z_B$ acting with Z only on B, use the provided `Z_ON_B`.

Calculate the density matrix for the flipped-bit case, probability-average the two cases, and return `rho_out`. Return a 4×4 NumPy array capable of representing complex numbers. Transforming a density matrix uses <strong>matrix multiplication</strong>, not elementwise array multiplication.

This function is an analysis tool for predicting the teleportation output. It receives `rho_target` for calculation and verification; it does not supply knowledge of the input state to Bob's actual correction circuit.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

In [ ]:
def noisy_transfer_density(rho_target, p):
    # TODO BEGIN
    # Z_ON_B is the 4×4 matrix applying Z only to B in R,B order.
    # Return the 4×4 density matrix including all trials.
    raise NotImplementedError("noisy_transfer_density")
    # TODO END

In [ ]:
def check_noise_density():
    fixtures = [
        (np.array([
            [1, -2j, 3, 1j],
            [2j, 4, 6j, -2],
            [3, -6j, 9, 3j],
            [-1j, -2, -3j, 1],
        ], dtype=complex) / 15, np.array([
            [[2, -4j, 6, 2j], [4j, 8, 12j, -4],
             [6, -12j, 18, 6j], [-2j, -4, -6j, 2]],
            [[2, -2j, 6, 1j], [2j, 8, 6j, -4],
             [6, -6j, 18, 3j], [-1j, -4, -3j, 2]],
            [[2, 0, 6, 0], [0, 8, 0, -4],
             [6, 0, 18, 0], [0, -4, 0, 2]],
            [[2, 4j, 6, -2j], [-4j, 8, -12j, -4],
             [6, 12j, 18, -6j], [2j, -4, 6j, 2]],
        ], dtype=complex) / 30),
        (np.array([
            [3, 2j, 1-1j, 1],
            [-2j, 5, 1j, 2],
            [1+1j, -1j, 3, 0],
            [1, 2, 0, 3],
        ], dtype=complex) / 14, np.array([
            [[6, 4j, 2-2j, 2], [-4j, 10, 2j, 4],
             [2+2j, -2j, 6, 0], [2, 4, 0, 6]],
            [[6, 2j, 2-2j, 1], [-2j, 10, 1j, 4],
             [2+2j, -1j, 6, 0], [1, 4, 0, 6]],
            [[6, 0, 2-2j, 0], [0, 10, 0, 4],
             [2+2j, 0, 6, 0], [0, 4, 0, 6]],
            [[6, -4j, 2-2j, -2], [4j, 10, -2j, 4],
             [2+2j, 2j, 6, 0], [-2, 4, 0, 6]],
        ], dtype=complex) / 28),
    ]
    for index, (rho, expected_outputs) in enumerate(fixtures):
        for p, expected in zip((0, 0.25, 0.5, 1), expected_outputs):
            result = np.asarray(noisy_transfer_density(rho.copy(), p))
            assert result.shape == (4, 4), "The returned density matrix must be 4×4."
            assert np.all(np.isfinite(result)), "Use finite entries in the density matrix."
            assert np.allclose(result, expected, atol=ATOL, rtol=0), (
                f"Test state {index}, p={p}: check the phase change on B and probability-weighted average.")

run_check("4a · Transmission-error output", check_noise_density)

#### (b) Compare Performance While Keeping the Original Correction
Use target $|\Phi^+\rangle_{RB}$ and calculate the output states at $p=0,1/4,1/2,1$. <strong>Keep the original correction rule unchanged in this part.</strong>

Complete the table using the provided expectation-value tools and `bell_fidelity()` from Problem 2.

| $p$ | $\langle XX\rangle$ | $\langle YY\rangle$ | $\langle ZZ\rangle$ | Joint fidelity |
|---:|---|---|---|---|
| $0$ | … | … | … | … |
| $1/4$ | … | … | … | … |
| $1/2$ | … | … | … | … |
| $1$ | … | … | … | … |

Also compare the joint Z-measurement distributions. Explain whether a Z-only check detects the error and what changes when X·Y correlations are included. Use the table to express fidelity under the original correction as a function of $p$.

The provided simulation performs teleportation including the cases in which the actual measurement bit is flipped in transmission. Check that your density matrix agrees with the simulation's exact output, then compare the samples from 4096 trials per measurement setting. Code checks use the exact output state.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

In [ ]:
def inspect_classical_bit_error():
    decoder = bob_decoder()
    table, results = [], {}
    for index, p in enumerate((0, 0.25, 0.5, 1)):
        rho = noisy_transfer_density(density(PHI_PLUS), p)
        reference = weighted_density(noisy_branches(decoder, p))
        assert np.allclose(rho, reference, atol=ATOL, rtol=0), "Compare the mixture expression with the actual branch calculation."
        exact = correlations(rho)
        sample = run_transfer_aer(decoder, p=p, keep="all", seed=SEED+400+index)
        sampled_E, sampled_F = sample_fidelity_table(sample)
        table.append([p] + list(exact.values()) + [bell_fidelity(exact), sampled_F])
        results[p] = {"rho": rho, "sample": sample, "exact": exact,
                      "F": bell_fidelity(exact), "sample_F": sampled_F}
    show_table(["p", "XX", "YY", "ZZ", "Exact joint F", "Aer estimated F"], table)
    show_table(["p", "P(00)", "P(01)", "P(10)", "P(11)"],
        [[p] + list(z_distribution(row["rho"]).values()) for p, row in results.items()])
    print("All trials are included. Only the received bit connected to c0 is flipped with probability p.")
    return results

_ = run_demo("4b", inspect_classical_bit_error,
    requires=("1b · Bob correction", "2b · Joint fidelity", "4a · Transmission-error output"))

#### (c) Final Comparison: Is Recovery Possible at $p=1$ and $p=1/2$?
You may now <strong>use the known error probability $p$ to modify Bob's correction rule</strong>. The goal is unchanged: for any input, restore the original joint state in every trial that can occur.

Bob can use the received $\widetilde m_Q,m_A$ and $p$. Apply operations only to B, using X, Z, and conditions on the received bits. You may not obtain Alice's original record again, act on R, use additional entanglement, or discard trials.

First fix Bob's received record to $(\widetilde m_Q,m_A)=(0,0)$. Complete the following table.

| Error probability | Possible original outcome $m_Q$ and error indicator $e$ | R,B state after X correction in each case | Can the required correction be determined from the received information? |
|---|---|---|---|
| $p=1$ | … | … | … |
| $p=1/2$ | … | … | … |

“Possible cases” means only cases with nonzero probability at that $p$. Check whether the same conclusion holds for the other received records.

<strong>If you conclude that recovery is possible,</strong> give a new correction table based on the two received bits and test it with the provided check cell. Note that c0 of the correction circuit now represents the received record rather than the original $m_Q$: it is <strong>$\widetilde m_Q$</strong>. The bit c1 is the correctly transmitted $m_A$.

The check uses both the Bell input and the relative-phase input from Problem 1, comparing each output with its appropriate target. Do not change the correction rule with the input. To verify the relative-phase input, use the provided general fidelity tool instead of `bell_fidelity()`, whose target is fixed to $|\Phi^+\rangle$.

<strong>If you conclude that recovery is impossible,</strong> show cases compatible with the same received record, and explain why Bob's available information and allowed operations cannot restore them all. Do not base the conclusion solely on the original circuit's low fidelity; use the possible joint states and their required corrections as evidence.

Finally, compare the two cases and explain in your own words <strong>why the probability of an incorrectly transmitted bit is a different question from whether the original measurement information can be recovered</strong>.

<strong>My Answer</strong>

Prediction and reasoning:

…

Results and interpretation:

…

<strong>Provided Experiment — Cases Compatible with the Same Received Record</strong>

Write your predictions in the table before running this cell. The original records shown here are for analysis and are not passed to Bob's correction function.

In [ ]:
def inspect_received_record():
    result = {}
    for p in (1, 0.5):
        # Inspect the state after X correction only. The original bit is visible only to the analysis tools.
        rows = [r for r in noisy_branches(missing_z_decoder(), p)
                if r["receivedQ"] == 0 and r["mA"] == 0 and r["probability"] > ATOL]
        total = sum(r["probability"] for r in rows)
        print(f"p={p}, Received record (Q,A)=(0,0)")
        show_table(["Original m_Q", "e", "Probability within this received record", "RB vector after X correction"],
            [[r["mQ"], r["e"], r["probability"]/total, vector_text(r["state"])] for r in rows])
        result[p] = rows
    return result

_ = run_demo("4c_evidence", inspect_received_record)

#### Test Your Proposed Correction Rule in Code

After completing the table and explanations, implement `recovery_plan(p)`. The function is tested with `p=1` and `p=0.5`.

- If you conclude that perfect recovery is possible in every trial, return a correction table keyed by the four received records. For each key `(received_mQ, mA)`, list the gates in execution order. For example, `("x", "z")` means X followed by Z, while `()` means no operation.
- If you conclude that perfect recovery is impossible, return `None` and explain your reasoning in the written field above.

The provided `decoder_from_table()` turns the table into an actual conditional circuit. The table must be independent of the input state and use only the received bits and known `p`. Code checks assess the decision and circuit behavior; the impossibility argument is reviewed separately.

In [ ]:
def recovery_plan(p):
    # TODO BEGIN
    # p is either 1 or 0.5.
    # If perfect recovery is possible, return an X/Z operation table for the four received records.
    # If you judge it impossible, return None and explain why in the written answer field.
    raise NotImplementedError("recovery_plan")
    # TODO END

In [ ]:
def check_recovery_plan():
    for p in (1.0, 0.5):
        plan = recovery_plan(p)
        if plan is None:
            rows = [row for row in noisy_branches(missing_z_decoder(), p, PHI_PLUS)
                    if row["probability"] > ATOL]
            conflicting_outputs = any(
                (left["receivedQ"], left["mA"]) == (right["receivedQ"], right["mA"])
                and not np.allclose(left["rho"], right["rho"], atol=ATOL, rtol=0)
                for index, left in enumerate(rows) for right in rows[index+1:]
            )
            assert conflicting_outputs, (
                f"p={p}: Check whether distinct outputs are compatible with the same received record.")
        else:
            decoder = decoder_from_table(plan)
            for index, psi in enumerate(probe_inputs(seed=417, count=3)):
                for row in noisy_branches(decoder, p, psi):
                    if row["probability"] > ATOL:
                        assert_scalar_close(fidelity(row["rho"], psi), 1.0,
                            f"p={p}: Input {index}, Received record ({row['receivedQ']},{row['mA']})")

run_check("4c · Recovery plan", check_recovery_plan)

In [ ]:
def inspect_recovery_plan():
    results = {}
    for p in (1, 0.5):
        plan = recovery_plan(p)
        if plan is None:
            print(f"p={p}: You concluded that perfect recovery is impossible. Also check the reasoning in your written answer.")
            results[p] = {"plan": None}
            continue
        decoder = decoder_from_table(plan)
        show_table(["Received Q", "Received A", "Correction in execution order"],
            [[a, b, plan[a, b] or "None"] for a, b in sorted(plan)])
        inputs = {}
        for index, (name, psi) in enumerate((("Bell input", PHI_PLUS), ("Relative-phase i input", PHASE_INPUT))):
            rows = [r for r in noisy_branches(decoder, p, psi) if r["probability"] > ATOL]
            show_table(["Input", "Original m_Q", "m_A", "Received Q", "e", "F with the correct target"],
                [[name, r["mQ"], r["mA"], r["receivedQ"], r["e"], fidelity(r["rho"], psi)] for r in rows])
            sample = run_transfer_aer(decoder, psi=psi, p=p, keep="all", seed=SEED+500+index)
            print(name, "Aer joint Z counts:", sample["ZZ"]["counts_all"])
            inputs[name] = {"branches": rows, "sample": sample}
        results[p] = {"plan": plan, "inputs": inputs}
    return results

_ = run_demo("4c", inspect_recovery_plan, requires=("4c · Recovery plan",))

## Final Application — What If the Other Measurement Bit Is Corrupted?

Now change <strong>only which bit is transmitted incorrectly</strong> in Problem 4. The input is $|\Phi^+\rangle_{RQ}$, the target output is $|\Phi^+\rangle_{RB}$, and the Bell resource and Alice's circuit remain unchanged.

The bit $m_Q$ is transmitted correctly, while only $m_A$ is flipped with probability $p$. Write the error indicator as $e_A$, equal to 1 for a flip and 0 otherwise. Then

$$
\widetilde m_A=m_A\oplus e_A,
\qquad \Pr(e_A=1)=p
$$

The error is independent of the input state and Alice's outcomes. Bob applies X according to the received $\widetilde m_A$, followed by Z according to the correctly transmitted $m_Q$. <strong>Keep the original correction rule and include every trial.</strong>

Compare the outputs with and without an error. Explain <strong>which residual operation remains on B</strong>, then write the overall output density matrix $\rho_{\mathrm{out}}^{(A)}(p)$. States that differ only by a global phase represent the same physical state.

Use your expression to complete the following row at $p=1/4$. Fidelity is still measured against $|\Phi^+\rangle_{RB}$.

| Corrupted record | $\langle XX\rangle$ | $\langle YY\rangle$ | $\langle ZZ\rangle$ | Joint fidelity |
|---|---|---|---|---|
| Only $m_A$, $p=1/4$ | … | … | … | … |

Compare with Problem 4(b) at the same $p$. In two or three sentences, explain <strong>which correlations differ between the error models and whether fidelity alone can distinguish them</strong>.

No new function or circuit is required. Answer with equations and the table first. If useful, check your density matrix with the provided `correlations()` and your `bell_fidelity()`. The earlier transmission-error experiment implements flips of $m_Q$, so it does not directly display results for this new condition.

<strong>Application Answer</strong>

Prediction and reasoning:

…

## Lab 1 Wrap-Up

Review the code-check statuses and make sure you have completed the written explanations. Finally, summarize in three sentences what must be checked when assessing the claim “teleportation succeeded.” Consider the input and target states, which trials are included, and the information Bob may use.

<strong>Wrap-Up Answer</strong>

…

In [ ]:
expected_checks = ["1b · Bob correction", "2b · Joint fidelity", "3b · Selection summary", "4a · Transmission-error output", "4c · Recovery plan"]
show_table(["Code check", "Status"], [[name, CHECK_STATUS.get(name, "Not run")] for name in expected_checks])
if all(CHECK_STATUS.get(name) == "Passed" for name in expected_checks):
    print("All five code checks passed. Review the written explanations separately.")
else:
    print("Review incomplete, unrun, or failed items. After revising an answer, rerun its check and subsequent experiments.")